# The Maximum-Margin Classifier & the `C` Hyperparameter

Companion to `4aMaximumMarginClassifier.md` and `4bSoftMarginAndC.md`.

We'll see, on the spam data:
1. the **margin** (the 'street') and the **support vectors**,
2. how the hyperparameter **`C`** widens / narrows it,
3. the **bias-variance** effect of `C`.

## 1. Load data, pick two features, standardise

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

PATH = 'Module Resources - SVM and Naive Bayes/SVM - Spam Classification/Spam.csv'
df = pd.read_csv(PATH)
f1, f2 = 'word_freq_your', 'word_freq_000'

# subsample for a clear, fast plot
sample = df.sample(400, random_state=0)
X = StandardScaler().fit_transform(sample[[f1, f2]].values)
y = sample['spam'].values
print('points:', X.shape[0], '| spam:', y.sum(), '| ham:', (y==0).sum())

## 2. A reusable plot: hyperplane + margin + support vectors

The dashed lines are the **margin edges** (`f = ±1`); the solid line is the
hyperplane (`f = 0`). Circled points are the **support vectors**.

In [ ]:
def plot_svm(clf, X, y, title):
    x_min,x_max = X[:,0].min()-1, X[:,0].max()+1
    y_min,y_max = X[:,1].min()-1, X[:,1].max()+1
    xx,yy = np.meshgrid(np.linspace(x_min,x_max,300), np.linspace(y_min,y_max,300))
    Z = clf.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    plt.figure(figsize=(7,6))
    plt.scatter(X[y==0,0], X[y==0,1], s=14, alpha=.5, color='steelblue', label='ham')
    plt.scatter(X[y==1,0], X[y==1,1], s=14, alpha=.5, color='crimson',  label='spam')
    # hyperplane (solid) and margins (dashed)
    plt.contour(xx,yy,Z, levels=[-1,0,1], colors='k',
                linestyles=['--','-','--'], linewidths=[1,2,1])
    # support vectors
    sv = clf.support_vectors_
    plt.scatter(sv[:,0], sv[:,1], s=90, facecolors='none', edgecolors='k',
                linewidths=1.2, label=f'support vectors ({len(sv)})')
    plt.xlabel(f1+' (scaled)'); plt.ylabel(f2+' (scaled)')
    plt.title(title); plt.legend(loc='upper right'); plt.show()

## 3. The margin and its support vectors (C = 1)

In [ ]:
clf = SVC(kernel='linear', C=1.0).fit(X, y)
plot_svm(clf, X, y, 'Linear SVM (C=1): hyperplane, margin, support vectors')
print('Number of support vectors:', len(clf.support_vectors_))
print('Training accuracy:', round(clf.score(X, y), 3))

## 4. The effect of `C` — wide vs narrow street

Small `C` → wide margin, many support vectors, tolerant.
Large `C` → narrow margin, few support vectors, strict (toward hard margin).

In [ ]:
for C in [0.01, 1, 100]:
    clf = SVC(kernel='linear', C=C).fit(X, y)
    plot_svm(clf, X, y, f'C = {C}  |  {len(clf.support_vectors_)} support vectors  '
                        f'|  train acc {clf.score(X,y):.3f}')

Watch how the **margin (dashed lines) narrows as C grows**, and the number of
support vectors drops. Big C hugs the data; small C keeps a wide, smooth street.

## 5. `C` as a bias-variance knob

Train accuracy vs cross-validated accuracy across a range of `C`.
Train keeps rising; CV peaks then falls — the classic bias-variance U (inverted).

In [ ]:
from sklearn.model_selection import cross_val_score

# use ALL data with the two features for a more stable curve
Xall = StandardScaler().fit_transform(df[[f1,f2]].values)
yall = df['spam'].values

Cs = [0.001, 0.01, 0.1, 1, 10, 100]
train_acc, cv_acc = [], []
for C in Cs:
    clf = SVC(kernel='linear', C=C).fit(Xall, yall)
    train_acc.append(clf.score(Xall, yall))
    cv_acc.append(cross_val_score(SVC(kernel='linear', C=C), Xall, yall, cv=5).mean())

plt.figure(figsize=(7,5))
plt.semilogx(Cs, train_acc, 'o-', label='train accuracy')
plt.semilogx(Cs, cv_acc,    's-', label='5-fold CV accuracy')
plt.xlabel('C (log scale)'); plt.ylabel('accuracy')
plt.title('C: small=underfit (high bias), large=overfit (high variance)')
plt.legend(); plt.grid(alpha=.3); plt.show()

for C,t,v in zip(Cs, train_acc, cv_acc):
    print(f'C={C:<7}  train={t:.3f}  cv={v:.3f}')

## Summary

```
• Margin = the street between the classes; SVM maximises its width.
• Support vectors = points on the margin edges; only they define the boundary.
• C controls the soft margin:
     small C → wide margin, more violations  → high bias, low variance (underfit)
     large C → narrow margin, few violations → low bias, high variance (overfit)
• Tune C (e.g. CV) to sit at the bias-variance sweet spot.
```

See [[4aMaximumMarginClassifier]], [[4bSoftMarginAndC]], [[1BiasVsVariance]].